In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Read from the raw source table created in setup
raw = spark.table("default.raw_retail_source")

bronze = (raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source",      lit("kaggle/carrie1/ecommerce-data"))
)

bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("default.bronze_retail")

print(f"Bronze rows: {spark.table('default.bronze_retail').count():,}")

In [0]:
spark.sql("DESCRIBE TABLE default.bronze_retail").show(truncate=False)
spark.sql("SELECT * FROM default.bronze_retail LIMIT 5").show(truncate=False)

# Check Delta history
spark.sql("DESCRIBE HISTORY default.bronze_retail").select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=False)

In [0]:
count = spark.table("default.bronze_retail").count()
dbutils.notebook.exit(f"bronze_rows={count:,}")